# Kitchen Assistant — Live End-to-End Demo

This notebook drives the **real** `app/` modules — no reimplementations, no copied logic — from the
session store all the way up to a **real Gemini Live session** talking through the real
`LiveGateway`.

| Section | What runs |
|---|---|
| 1. Session state | `app/state_manager.py` — the single store every tool writes through |
| 2. Recipe catalog (RAG) | `app/services/recipe_store.py` — DuckDB + `gemini-embedding-001` (**live API call**) |
| 3. Tool registry | `app/tools/registry.py` — the exact `FunctionDeclaration`s handed to Gemini |
| 4. Tool chain | `registry.dispatch(...)` — the same entry point `LiveGateway._handle_tool_call` uses |
| 5. Timer engine | `app/services/timer_engine.py` — a real `asyncio` countdown + proactive expiry callback |
| 6. Live session | `app/live/gateway.py` — real `client.aio.live.connect`, real tool call, real audio back |

**Honesty note.** Sections 2 and 6 make real network calls and need `GOOGLE_API_KEY` in `.env`.
Both cells check for the key first: without one they print a clearly-labelled `[SKIPPED - no
GOOGLE_API_KEY]` line and the rest of the notebook still runs. Nothing here fabricates model
output — if you see a transcript below, a model produced it.

## 0. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "app").exists() else Path.cwd().parent
assert (ROOT / "app").exists(), f"Could not locate the repo root from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

import asyncio
import json
import os

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from app.services.recipe_store import RecipeStore
from app.services.timer_engine import TimerEngine
from app.state_manager import StateManager
from app.tools import cooking_tools
from app.tools.registry import ToolRegistry

HAS_KEY = bool(os.getenv("GOOGLE_API_KEY"))
LIVE_MODEL = os.getenv("LIVE_MODEL", "gemini-3.1-flash-live-preview")


def show(title, payload):
    print(f"{title} ->")
    print(json.dumps(payload, indent=2, default=str))
    print()


def skipped(what):
    print(f"[SKIPPED - no GOOGLE_API_KEY in .env] {what}")


print("repo root :", ROOT)
print("API key   :", "present" if HAS_KEY else "absent (live sections will skip)")
print("live model:", LIVE_MODEL)

repo root : C:\Users\thoma\Downloads\kitchen-assistant
API key   : present
live model: gemini-3.1-flash-live-preview


## 1. Session state

`StateManager` (`app/state_manager.py`) is the single store — ADR-003. Every read-modify-write goes
through `update()`, which serializes concurrent mutations of one session behind a per-session
`asyncio.Lock`. A fresh in-memory instance here mirrors the app default (`USE_REDIS=false`).

In [2]:
state_manager = StateManager(use_redis=False)
timer_engine = TimerEngine(state_manager)
recipe_store = RecipeStore(db_path=str(ROOT / "data" / "recipes.db"))
registry = ToolRegistry(state_manager, timer_engine=timer_engine, recipe_store=recipe_store)

session_id = "notebook-demo"
state = await state_manager.get_or_create_state(session_id)
print(state.model_dump_json(indent=2))

{
  "session_id": "notebook-demo",
  "recipe_id": null,
  "recipe_metadata": null,
  "current_step_index": 0,
  "active_timers": {},
  "servings_multiplier": 1.0,
  "last_updated": "2026-07-25T18:25:02.526532"
}


## 2. Recipe catalog — semantic search over DuckDB

`RecipeStore.search()` embeds the query with `gemini-embedding-001` (**live call**) and ranks the
catalog with `array_distance` over a `FLOAT[3072]` column in `data/recipes.db` (ADR-004).
`get_recipe()` is pure DuckDB and needs no key, so it runs either way.

In [3]:
if HAS_KEY:
    hits = await cooking_tools.search_recipes(
        recipe_store, query="something creamy with mushrooms", k=3
    )
    show("search_recipes('something creamy with mushrooms')", hits)
    top_id = hits["results"][0]["id"]
else:
    skipped("search_recipes needs an embedding call; falling back to a fixed recipe id.")
    top_id = "r2"

recipe = await recipe_store.get_recipe(top_id)   # pure DuckDB, no API key needed
print(f"get_recipe({top_id!r}) -> {recipe.title}  ({recipe.total_time_minutes} min, "
      f"{len(recipe.ingredients)} ingredients, {len(recipe.steps)} steps)")

search_recipes('something creamy with mushrooms') ->
{
  "status": "success",
  "results": [
    {
      "id": "r15",
      "title": "Creamy Tomato Basil Soup",
      "total_time_minutes": 25
    },
    {
      "id": "r2",
      "title": "Mushroom Risotto",
      "total_time_minutes": 45
    },
    {
      "id": "r6",
      "title": "Chicken Tikka Masala",
      "total_time_minutes": 50
    }
  ]
}

get_recipe('r15') -> Creamy Tomato Basil Soup  (25 min, 5 ingredients, 3 steps)


## 3. The tool registry — exactly what Gemini is shown

`ToolRegistry` pairs each `types.FunctionDeclaration` with an async callable. Note what the
declarations do **not** contain: `session_id`, `state_manager`, `timer_engine` and `recipe_store`
are injected server-side at dispatch time by signature inspection, so the model can never address
another chef's session.

In [4]:
print("registered tools:")
for declaration in registry.declarations:
    params = list((declaration.parameters.properties or {}).keys())
    print(f"  {declaration.name:<20} {params}")

print("\none full declaration, as sent in LiveConnectConfig(tools=...):")
print(registry.declarations[0].model_dump_json(indent=2, exclude_none=True))

print("\ninjected server-side, never model-visible:",
      ["session_id", "state_manager", "timer_engine", "recipe_store"])

registered tools:
  set_kitchen_timer    ['duration_seconds', 'label']
  cancel_timer         ['timer_id']
  list_timers          []
  convert_units        ['value', 'from_unit', 'to_unit']
  scale_recipe         ['multiplier']
  search_recipes       ['query', 'k']
  load_recipe          ['recipe_id']
  navigate_steps       ['direction', 'step_index']

one full declaration, as sent in LiveConnectConfig(tools=...):
{
  "description": "Set a countdown kitchen timer with a descriptive label.",
  "name": "set_kitchen_timer",
  "parameters": {
    "properties": {
      "duration_seconds": {
        "description": "Timer length in seconds. Must be positive.",
        "type": "INTEGER"
      },
      "label": {
        "description": "Short name for the timer, e.g. 'pasta' or 'roast'.",
        "type": "STRING"
      }
    },
    "required": [
      "duration_seconds",
      "label"
    ],
    "type": "OBJECT"
  }
}

injected server-side, never model-visible: ['session_id', 'state_manager', '

## 4. A full tool chain through `dispatch()`

`registry.dispatch(session_id, name, args)` is exactly what `LiveGateway._handle_tool_call` calls
when Gemini emits a `tool_call`. Below is the sequence a real voice turn produces — load, scale,
navigate, convert, timer — with each result printed the way the model receives it.

In [5]:
chain_session = "notebook-chain"

loaded = await registry.dispatch(chain_session, "load_recipe", {"recipe_id": top_id})
show("load_recipe", loaded)

scaled = await registry.dispatch(chain_session, "scale_recipe", {"multiplier": 2.5})
show("scale_recipe(2.5)", scaled)

step = await registry.dispatch(chain_session, "navigate_steps", {"direction": "next"})
show("navigate_steps('next')", step)

conv = await registry.dispatch(
    chain_session, "convert_units", {"value": 2, "from_unit": "cups", "to_unit": "ml"}
)
show("convert_units(2 cups -> ml)", conv)

bad = await registry.dispatch(
    chain_session, "convert_units", {"value": 1, "from_unit": "lb", "to_unit": "ml"}
)
show("convert_units(lb -> ml)  # mass<->volume is refused, not guessed", bad)

timer = await registry.dispatch(
    chain_session, "set_kitchen_timer", {"duration_seconds": 540, "label": "pasta"}
)
show("set_kitchen_timer(540s, 'pasta')", timer)

show("list_timers", await registry.dispatch(chain_session, "list_timers", {}))

load_recipe ->
{
  "status": "success",
  "recipe_id": "r15",
  "title": "Creamy Tomato Basil Soup",
  "total_steps": 3,
  "first_instruction": "Saut\u00e9 garlic until fragrant, then add tomatoes and broth."
}

scale_recipe(2.5) ->
{
  "status": "success",
  "new_multiplier": 2.5,
  "scaled_ingredients": [
    {
      "name": "Canned Tomatoes",
      "amount": 2000.0,
      "unit": "g"
    },
    {
      "name": "Vegetable Broth",
      "amount": 1250.0,
      "unit": "ml"
    },
    {
      "name": "Heavy Cream",
      "amount": 250.0,
      "unit": "ml"
    },
    {
      "name": "Basil Leaves",
      "amount": 25.0,
      "unit": "piece"
    },
    {
      "name": "Garlic",
      "amount": 7.5,
      "unit": "clove"
    }
  ]
}

navigate_steps('next') ->
{
  "status": "success",
  "current_step": 1,
  "total_steps": 3,
  "instruction": "Simmer 15 minutes, then blend until smooth."
}

convert_units(2 cups -> ml) ->
{
  "status": "success",
  "value": 473.18,
  "unit": "ml"
}

conver

In [6]:
final = await state_manager.get_state(chain_session)
print("session state after the chain — this is what ships to the browser as state.snapshot:")
print(final.model_dump_json(indent=2)[:1400], "...")

session state after the chain — this is what ships to the browser as state.snapshot:
{
  "session_id": "notebook-chain",
  "recipe_id": "r15",
  "recipe_metadata": {
    "id": "r15",
    "title": "Creamy Tomato Basil Soup",
    "description": null,
    "ingredients": [
      {
        "name": "Canned Tomatoes",
        "amount": 800.0,
        "unit": "g"
      },
      {
        "name": "Vegetable Broth",
        "amount": 500.0,
        "unit": "ml"
      },
      {
        "name": "Heavy Cream",
        "amount": 100.0,
        "unit": "ml"
      },
      {
        "name": "Basil Leaves",
        "amount": 10.0,
        "unit": "piece"
      },
      {
        "name": "Garlic",
        "amount": 3.0,
        "unit": "clove"
      }
    ],
    "steps": [
      {
        "step_number": 1,
        "instruction": "Sauté garlic until fragrant, then add tomatoes and broth.",
        "duration_minutes": null
      },
      {
        "step_number": 2,
        "instruction": "Simmer 15 minut

## 5. Timer engine — a real countdown, really expiring

`TimerEngine` owns actual `asyncio` tasks keyed by `(session_id, timer_id)`. The gateway registers a
per-session callback at connect time so an expiry can be announced *unprompted* — the engine never
imports the gateway, so there is no cycle. Here a notebook callback stands in for the gateway's, and
we wait out a real 5-second timer.

In [7]:
expired = asyncio.Event()


async def on_expiry(timer):
    print(f"[proactive callback] timer '{timer.label}' expired -> the gateway would emit "
          "timer.expired to the browser and nudge the model to announce it")
    expired.set()


timer_engine.register_session(chain_session, on_expiry)

quick = await cooking_tools.set_kitchen_timer(
    state_manager, chain_session, duration_seconds=5, label="quick demo",
    timer_engine=timer_engine,
)
show("set_kitchen_timer(5s, 'quick demo')", quick)
print("active countdown tasks:", timer_engine.active_count(chain_session))

await asyncio.wait_for(expired.wait(), timeout=15)

state_after = await state_manager.get_state(chain_session)
print("timer marked inactive in state:",
      not state_after.active_timers[quick["timer_id"]].is_active)

set_kitchen_timer(5s, 'quick demo') ->
{
  "status": "success",
  "timer_id": "7da91d3f",
  "label": "quick demo",
  "duration": 5
}

active countdown tasks: 2


[proactive callback] timer 'quick demo' expired -> the gateway would emit timer.expired to the browser and nudge the model to announce it
timer marked inactive in state: True


## 6. A real Gemini Live session, through the real gateway

This is the end-to-end path. `LiveGateway` is constructed exactly as `app/main.py` does — same state
manager, same registry, same timer engine — and uses its **real** connect factory
(`client.aio.live.connect`). The only substitution is the browser: a `ScriptedWebSocket` feeds one
`user.text` envelope in and records every frame the gateway sends back, which is precisely what
`static/app.js` and the React HUD receive over the wire.

Everything on this path is real: the Live session, the model's decision to call a tool, the
server-side dispatch, the tool response, and the PCM16 audio reply.

In [8]:
from app.live.gateway import LiveGateway


class ScriptedWebSocket:
    """Stands in for the browser end of /ws/voice/{session_id}.

    Speaks the ASGI message shape LiveGateway._uplink expects, replays a
    scripted list of client envelopes, and records everything sent back.
    """

    def __init__(self, script):
        self._script = list(script)
        self.sent_text: list[str] = []
        self.audio = bytearray()
        self.last_frame_at = 0.0

    async def receive(self):
        if self._script:
            await asyncio.sleep(0.3)
            return {"type": "websocket.receive", "text": json.dumps(self._script.pop(0))}
        await asyncio.sleep(3600)   # idle: the downlink drives the rest of the turn

    async def send_text(self, text):
        self.sent_text.append(text)
        self.last_frame_at = asyncio.get_running_loop().time()

    async def send_bytes(self, data):
        self.audio.extend(data)
        self.last_frame_at = asyncio.get_running_loop().time()


async def run_live_turn(prompt, quiet_for=3.0, deadline=60.0):
    """Drive one conversational turn, then stop once the model has gone quiet."""
    ws = ScriptedWebSocket([{"type": "user.text", "text": prompt}])
    gateway = LiveGateway(
        websocket=ws,
        session_id="notebook-live",
        state_manager=state_manager,
        registry=registry,
        timer_engine=timer_engine,
    )
    task = asyncio.create_task(gateway.run())
    loop = asyncio.get_running_loop()
    started = loop.time()
    while loop.time() - started < deadline:
        await asyncio.sleep(0.5)
        if task.done():
            break
        if ws.last_frame_at and ws.audio and loop.time() - ws.last_frame_at > quiet_for:
            break
    gateway._closing = True
    task.cancel()
    await asyncio.gather(task, return_exceptions=True)
    return ws


PROMPT = "Set a timer for nine minutes labelled pasta, then tell me how many timers are running."

if HAS_KEY:
    ws = await run_live_turn(PROMPT)
    print(f"chef (typed): {PROMPT}\n")
    print(f"frames back to the browser: {len(ws.sent_text)} JSON, "
          f"{len(ws.audio):,} bytes of PCM16 24 kHz audio "
          f"(~{len(ws.audio) / 2 / 24000:.1f}s of speech)\n")
    for raw in ws.sent_text:
        envelope = json.loads(raw)
        if envelope["type"] == "state.snapshot":
            timers = envelope["state"]["active_timers"].values()
            print(f"  {'state.snapshot':<16} active_timers="
                  f"{[t['label'] for t in timers]}")
        elif envelope["type"].startswith("transcript"):
            print(f"  {envelope['type']:<16} {envelope['text']!r}")
        else:
            print(f"  {envelope['type']:<16} {json.dumps(envelope)[:90]}")
else:
    ws = None
    skipped("section 6 needs a real Gemini Live session.")

chef (typed): Set a timer for nine minutes labelled pasta, then tell me how many timers are running.

frames back to the browser: 10 JSON, 190,562 bytes of PCM16 24 kHz audio (~4.0s of speech)

  session.status   {"type": "session.status", "status": "ready"}
  state.snapshot   active_timers=['pasta']
  transcript.agent 'Timer set.'
  transcript.agent ' One'
  transcript.agent ' timer'
  transcript.agent ' running.'
  transcript.agent ' "pasta"'
  transcript.agent ' (540s'
  transcript.agent ' remaining).'
  session.status   {"type": "session.status", "status": "reconnecting"}


### What just happened

The model was never told about `session_id` — it asked for `set_kitchen_timer(duration_seconds=540,
label='pasta')`, the gateway injected the session, `cooking_tools.set_kitchen_timer` wrote through
`StateManager`, `TimerEngine` started a real countdown, and the result went back to the model as a
`FunctionResponse`. The `state.snapshot` frame above is what repaints the HUD's timer board.

The audio is raw PCM16 mono at 24 kHz — the browser plays it straight into an
`AudioContext({sampleRate: 24000})` with no decoding step. Saved below as a WAV so you can listen to
the actual reply.

The trailing `session.status: reconnecting` frame is not an error — Live connections are time-limited, so when the session ends the gateway transparently reconnects with its stored resumption handle (ADR-005) and the browser only ever sees a status change.

In [9]:
import wave

if ws is not None and ws.audio:
    out = ROOT / "assets" / "live_demo_reply.wav"
    out.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(out), "wb") as handle:
        handle.setnchannels(1)
        handle.setsampwidth(2)
        handle.setframerate(24000)
        handle.writeframes(bytes(ws.audio))
    print(f"wrote {out.relative_to(ROOT)}  ({out.stat().st_size:,} bytes)")
else:
    skipped("no audio captured, so no WAV was written.")

wrote assets\live_demo_reply.wav  (190,606 bytes)


## What this notebook does *not* cover

- **Microphone capture and playback.** The `AudioWorklet` uplink (16 kHz) and the 24 kHz playback
  queue live only in `static/app.js` and `frontend/src/hooks/useVoiceSocket.ts`. Run
  `uvicorn app.main:app --reload` and open `http://localhost:8000/` to exercise them.
- **Camera doneness checks.** `video.frame` envelopes need a real camera; the gateway's forwarding
  path is covered in `tests/test_gateway.py`.
- **Barge-in and GoAway reconnection.** Both need a long session or a mid-sentence interruption;
  `tests/test_gateway.py` drives them against a fake Live backend instead.
- **The latency budget.** The <800 ms glass-to-glass target in `ARCHITECTURE.md` is a property of
  the mic-to-speaker path, not of the text-in turn above.

`poetry run pytest` covers all of the above with no API key required.